In [ ]:
%%javascript(() => {  // 只隐藏编辑器，不隐藏 cell 的 toolbar / prompt  const selectors = ['.jp-InputArea-editor', '.cm-editor', '.CodeMirror'];  // 找到当前 Notebook 使用的编辑器 DOM（优先匹配第一个存在的 selector）  function getEditors() {    for (const s of selectors) {      const nodes = document.querySelectorAll(s);      if (nodes.length) return { sel: s, nodes };    }    return { sel: null, nodes: [] };  }  // 切换显示/隐藏  function toggle() {    const { sel, nodes } = getEditors();    if (!nodes.length) return alert('没找到编辑器区域：' + selectors.join(' / '));    const hide = nodes[0].style.display !== 'none';    nodes.forEach(n => n.style.display = hide ? 'none' : '');    // 仅用于调试：输出当前使用的 selector    console.log("toggle selector:", sel);  }  // 创建右上角按钮（避免重复创建）  let btn = document.getElementById('toggleCodeBtn');  if (!btn) {    btn = document.createElement('button');    btn.id = 'toggleCodeBtn';    btn.textContent = 'Hide/Show Code';    btn.style.cssText =      'position:fixed;top:12px;right:12px;z-index:99999;padding:6px 12px;border-radius:6px;';    btn.addEventListener('click', toggle);    document.body.appendChild(btn);  }  // 默认隐藏编辑器（只隐藏代码，不影响按钮/工具栏/输出）  const { nodes } = getEditors();  nodes.forEach(n => n.style.display = 'none');})();

# 风险评估分析

**目标**: 计算市场风险暴露得分，提供风险预警

**风险指标权重**:
- 波动率分位数: 50%
- 估值偏离度: 20%
- 情绪极端度: 15%
- 外部风险: 15%

In [ ]:
# 统一环境初始化（自动检测项目路径）from notebooks.lib import (    setup_research_environment,    ErrorBoundary,    ResultSaver)# 初始化研究环境env = setup_research_environment(verbose=True)# 导入必要的库import pandas as pdimport numpy as npfrom datetime import datetime, timedelta# 从环境获取组件jq = Nonetry:    jq = env.get_jqdata_client()except Exception as e:    print(f"⚠️ JQData初始化失败: {{e}}")# 加载配置config = env.load_config('config')INDEX_CODE = config.get('data', {{}}).get('default_index', '000001.XSHG')# 初始化结果保存器result_saver = ResultSaver("risk_assessment")print('✅ 环境加载完成')

## 1. 当前风险评估

In [ ]:
index_code = "000001.XSHG"
eval_result = evaluator.evaluate(index_code=index_code)

print("=" * 60)
print("风险评估结果")
print("=" * 60)
print(f"评估日期: {eval_result.evaluation_date}")
print(f"风险得分: {eval_result.risk_score:.1f}/100")

# 风险等级判断
if eval_result.risk_score >= 80:
    risk_level = "🔴 极高风险"
elif eval_result.risk_score >= 60:
    risk_level = "🟠 高风险"
elif eval_result.risk_score >= 40:
    risk_level = "🟡 中等风险"
else:
    risk_level = "🟢 低风险"

print(f"风险等级: {risk_level}")

## 2. 波动率分析

In [ ]:
# 获取历史数据计算波动率
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

df = jq.get_price(index_code, start_date=start_date, end_date=end_date,
                  frequency='daily', fields=['close'])
df = df.reset_index()
df.columns = ['date', 'close']

# 计算日收益率
df['returns'] = df['close'].pct_change()

# 计算滚动波动率
df['volatility_20'] = df['returns'].rolling(20).std() * np.sqrt(252) * 100
df['volatility_60'] = df['returns'].rolling(60).std() * np.sqrt(252) * 100

# 当前波动率
current_vol_20 = df['volatility_20'].iloc[-1]
current_vol_60 = df['volatility_60'].iloc[-1]

# 波动率分位数
vol_percentile = (df['volatility_20'].dropna() < current_vol_20).mean() * 100

print(f"\n📊 波动率分析:")
print(f"  20日波动率: {current_vol_20:.2f}%")
print(f"  60日波动率: {current_vol_60:.2f}%")
print(f"  当前波动率分位数: {vol_percentile:.1f}%")

In [ ]:
# 波动率可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 价格图
ax1 = axes[0]
ax1.plot(df['date'], df['close'], color='blue', linewidth=1)
ax1.set_title(f'{index_code} 价格走势')
ax1.grid(True, alpha=0.3)

# 波动率图
ax2 = axes[1]
ax2.plot(df['date'], df['volatility_20'], label='20日波动率', color='orange')
ax2.plot(df['date'], df['volatility_60'], label='60日波动率', color='red')
ax2.axhline(y=20, color='green', linestyle='--', alpha=0.5, label='低波动阈值')
ax2.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='高波动阈值')
ax2.set_title('年化波动率')
ax2.set_ylabel('%')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 风险来源分解

In [ ]:
# 计算各风险来源
risk_components = {}

# 1. 趋势风险：负向趋势增加风险
trend_risk = 0
if eval_result.trend_score < -0.5:
    trend_risk = 30
elif eval_result.trend_score < -0.2:
    trend_risk = 15
elif eval_result.trend_score < 0:
    trend_risk = 5
risk_components['趋势风险'] = trend_risk

# 2. 市场环境风险
regime_risk = 0
if eval_result.market_regime == 'bear':
    regime_risk = 30
elif eval_result.market_regime == 'volatile':
    regime_risk = 15
risk_components['环境风险'] = regime_risk

# 3. 分布日风险
dist_risk = 0
if eval_result.ibd_result and eval_result.ibd_result.distribution_count >= 3:
    dist_risk = min(20, eval_result.ibd_result.distribution_count * 5)
risk_components['分布日风险'] = dist_risk

# 4. 波动率风险
vol_risk = 0
if vol_percentile > 80:
    vol_risk = 20
elif vol_percentile > 60:
    vol_risk = 10
risk_components['波动率风险'] = vol_risk

print("\n🔍 风险来源分解:")
for name, value in risk_components.items():
    bar = '█' * int(value / 2) + '░' * (15 - int(value / 2))
    print(f"  {name}: {bar} {value}")

total_risk = sum(risk_components.values())
print(f"\n  总计: {total_risk} (基础值50)")

## 4. 风险预警机制

In [ ]:
warnings = []

# 检查各项风险
if eval_result.risk_score >= 70:
    warnings.append("⚠️ 综合风险得分较高，建议降低仓位")

if vol_percentile > 80:
    warnings.append("⚠️ 当前波动率处于历史高位，注意控制仓位")

if eval_result.ibd_result and eval_result.ibd_result.distribution_count >= 4:
    warnings.append(f"⚠️ 分布日数量达{eval_result.ibd_result.distribution_count}个，机构可能在出货")

if eval_result.trend_score < -0.3:
    warnings.append("⚠️ 趋势转弱，建议谨慎操作")

if eval_result.reversal_signal < -0.5:
    warnings.append("⚠️ 出现强烈看跌反转信号")

print("\n🚨 风险预警:")
if warnings:
    for w in warnings:
        print(f"  {w}")
else:
    print("  ✅ 暂无重大风险预警")

## 5. 仓位建议

In [ ]:
from core.dynamic_signals import suggested_position_ratio

position = suggested_position_ratio()

print("\n💰 仓位建议:")
print(f"  建议仓位: {position:.1%}")

if position >= 0.8:
    print("  策略: 进攻型配置，可适当加仓")
elif position >= 0.6:
    print("  策略: 稳健型配置，维持正常仓位")
elif position >= 0.4:
    print("  策略: 防御型配置，控制仓位")
else:
    print("  策略: 保守型配置，降低仓位观望")

## 6. 保存研究结论

In [ ]:
conclusion = {
    "evaluation_date": eval_result.evaluation_date,
    "risk_score": eval_result.risk_score,
    "risk_components": risk_components,
    "volatility_20d": current_vol_20,
    "volatility_percentile": vol_percentile,
    "suggested_position": position,
    "warnings": warnings
}

save_research_conclusion(
    module="risk_assessment",
    findings=conclusion,
    recommendation=f"风险得分: {eval_result.risk_score:.1f}/100, 建议仓位: {position:.1%}",
    metadata={"index_code": index_code}
)

print("\n✅ 研究结论已保存")